# ¿Cómo cambia la asociación al adelantar la serie solar?
Cada par compara SSN(t) con la variable en t+k. El desfase positivo se mide en meses.
Ejecuta las celdas en orden. Las tablas CSV y figuras se guardan en la carpeta de trabajo.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]


In [2]:
filas = []
for variable in variables:
    for meses in desfases:
        pares = pd.concat([df.SSN, df[variable].shift(-meses)], axis=1).dropna()
        filas.append([variable, meses, len(pares), pares.corr().iloc[0,1],
                      pares.corr(method="spearman").iloc[0,1]])
tabla = pd.DataFrame(filas, columns=["variable","meses","pares","Pearson","Spearman"])
tabla.to_csv("11_desfases.csv", index=False)
print(tabla.round(3).to_string(index=False))
fig, ejes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
for ax, metodo in zip(ejes, ["Pearson","Spearman"]):
    tabla.pivot(index="meses", columns="variable", values=metodo).rename(columns={"precipitacion_mm":"Lluvia"}).plot(ax=ax, marker="o")
    ax.set(title=metodo, xlabel="Desfase (meses)", xticks=desfases, ylabel="Correlación")
    ax.axhline(0, color="gray", lw=1)
fig.tight_layout()
fig.savefig("11_desfases.png", dpi=180)
plt.show()


        variable  meses  pares  Pearson  Spearman
            ICEN      0    917    0.092     0.120
            ICEN      3    914    0.125     0.162
            ICEN      6    911    0.143     0.180
            ICEN     12    905    0.123     0.159
            ICEN     24    893    0.120     0.171
             ONI      0    917    0.081     0.085
             ONI      3    914    0.125     0.119
             ONI      6    911    0.150     0.143
             ONI     12    905    0.180     0.172
             ONI     24    893    0.187     0.230
precipitacion_mm      0    917   -0.056    -0.064
precipitacion_mm      3    914   -0.018    -0.042
precipitacion_mm      6    911    0.007    -0.009
precipitacion_mm     12    905   -0.029    -0.041
precipitacion_mm     24    893    0.011     0.002
